# Impute the missing data
Treat the records without any data (no-admission) with masking


In [ ]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

from data_preprocess.impute_fn import impute_ts_ehr_bffill, impute_ts_ehr_linear_interp, impute_ts_ehr_global

In [ ]:
# load data
time_resolution = "2h"
interested_split = 'train'
impute_method = "bffill" # choose from bffill, global, linearinterp

if time_resolution == "2h":
    num_timestep = 24
elif time_resolution == "1h":
    num_timestep = 48
elif time_resolution == "30min":
    num_timestep = 96
else:
    raise ValueError(f"Unknown time resolution {time_resolution}")

split_dir = f"data/MIMICIII_last48h_ts{time_resolution}/split"
if impute_method != "bffill": # bffill is the default option
    export_dir = f"data/MIMICIII_last48h_ts{time_resolution}_impute-{impute_method}/imputed"
else:
    export_dir = f"data/MIMICIII_last48h_ts{time_resolution}/imputed"

Path(os.path.join(export_dir, interested_split)).mkdir(exist_ok=True, parents=True)
train_ts_path = os.path.join(split_dir, 'train', 'time-series.csv')

demo_path = os.path.join(split_dir, interested_split, 'demographics.csv')
ts_path = os.path.join(split_dir, interested_split, 'time-series.csv')
label_path = os.path.join(split_dir, interested_split, "label.csv")
demo_export_path = os.path.join(export_dir, interested_split, 'demographics.csv')
ts_export_path = os.path.join(export_dir, interested_split, 'time-series.csv')
label_export_path = os.path.join(export_dir, interested_split, "label.csv")

train_ts_df = pd.read_csv(train_ts_path)

demo_df = pd.read_csv(demo_path)
ts_df = pd.read_csv(ts_path)
label_df = pd.read_csv(label_path)

print(f"hadm_id num of demo_df:{demo_df['hadm_id'].unique().shape[0]}")
print(f"hadm_id num of vital_df:{ts_df['hadm_id'].unique().shape[0]}")
print(f"hadm_id num of label_df:{label_df['hadm_id'].unique().shape[0]}")

## Get the wanted features

First fill all the empty timestep with nan

In [ ]:
used_hadm_id = demo_df['hadm_id'].unique()
ts_df = ts_df[ts_df['hadm_id'].isin(used_hadm_id)]
label_df = label_df[label_df['hadm_id'].isin(used_hadm_id)]

used_hadm_id = ts_df['hadm_id'].unique()
demo_df = demo_df[demo_df['hadm_id'].isin(used_hadm_id)]
label_df = label_df[label_df['hadm_id'].isin(used_hadm_id)]

used_hadm_id = label_df['hadm_id'].unique()
demo_df = demo_df[demo_df['hadm_id'].isin(used_hadm_id)]
ts_df = ts_df[ts_df['hadm_id'].isin(used_hadm_id)]

features_columns = [
    'heartrate_min', 'heartrate_max', 'heartrate_mean',
    'sysbp_min', 'sysbp_max', 'sysbp_mean',
    'diasbp_min', 'diasbp_max', 'diasbp_mean',
    'meanbp_min', 'meanbp_max', 'meanbp_mean',
    'resprate_min', 'resprate_max', 'resprate_mean',
    'tempc_min', 'tempc_max', 'tempc_mean',
    'spo2_min', 'spo2_max', 'spo2_mean',
    'glucose_min', 'glucose_max', 'glucose_mean',
    'ALBUMIN', 'ANION GAP', 'BANDS', 'BICARBONATE',
    'BILIRUBIN', 'BUN', 'CHLORIDE', 'CREATININE',
    'GLUCOSE', 'HEMATOCRIT', 'HEMOGLOBIN', 'INR',
    'LACTATE', 'PLATELET', 'POTASSIUM', 'PT', 'PTT', 'SODIUM', 'WBC'
]

print(f"hadm_id num of demo_df:{demo_df['hadm_id'].unique().shape[0]}")
print(f"hadm_id num of vital_df:{ts_df['hadm_id'].unique().shape[0]}")
print(f"hadm_id num of label_df:{label_df['hadm_id'].unique().shape[0]}")


In [ ]:
# Fill missing timesteps
unique_hadm_ids = ts_df[["subject_id", "hadm_id"]].drop_duplicates()
timepoints = pd.DataFrame({"timepoint": range(num_timestep)}) 

complete_timepoints = (
    unique_hadm_ids.merge(timepoints, how="cross")
)

ts_df = complete_timepoints.merge(ts_df, on=["subject_id", "hadm_id", "timepoint"], how="left")

print(ts_df["timepoint"].value_counts())


## Process missing value

In [ ]:
# missing value imputation
if impute_method == "bffill":
    ts_df, overall_dict = impute_ts_ehr_bffill(ts_df, features_columns, train_ts_df=train_ts_df)
elif impute_method == "linearinterp":
    ts_df, overall_dict = impute_ts_ehr_linear_interp(ts_df, features_columns, train_ts_df=train_ts_df)
elif impute_method == "global":
    ts_df, overall_dict = impute_ts_ehr_global(ts_df, features_columns, train_ts_df=train_ts_df)
else:
    raise ValueError(f"Unknown impute method {impute_method}")

## Process the ethnic and gender feature

In [ ]:
# Ethnicity before processing
demo_df["ethnicity"].unique()

In [ ]:
def categorize_ethnicity(ethnicity):
    if any(keyword in ethnicity for keyword in ["WHITE"]):
        return "White"
    elif any(keyword in ethnicity for keyword in ["ASIAN"]):
        return "Asian"
    elif any(keyword in ethnicity for keyword in ["BLACK"]):
        return "Black"
    elif any(keyword in ethnicity for keyword in ["HISPANIC", "LATINO"]):
        return "Hispanic"
    elif ethnicity in ["UNABLE TO OBTAIN", "UNKNOWN/NOT SPECIFIED", "PATIENT DECLINED TO ANSWER"]:
        return "Unknown"
    else:
        return "Other"

demo_df['ethnicity'] = demo_df['ethnicity'].apply(categorize_ethnicity)

print(demo_df["ethnicity"].unique())
print(demo_df["gender"].unique())

In [ ]:
ethnic_map = {"White": 0, "Black": 1, "Asian": 2, "Hispanic": 3, "Other": 4, "Unknown": 5}
demo_df["ethnicity_category"] = demo_df["ethnicity"].map(ethnic_map)

gender_map = {"F":0, "M":1}
demo_df["gender_category"] = demo_df["gender"].map(gender_map)

demo_df.head()

## Save data

In [ ]:
# save dfs
ts_df.to_csv(ts_export_path, index=False, sep=',')
demo_df.to_csv(demo_export_path, index=False, sep=',')
label_df.to_csv(label_export_path, index=False, sep=",")